## Coding Homework 5 - Chapter 11

### Double-click to add your name here and edit this Markdown cell

### 3/15/2026

<mark>_____________________________________________________________________________________</mark>

## Coding Tasks Description

The purpose of this coding homework is to practice some coding with hash maps as discussed in chapter 11 of the CLRS textbook.


**Tasks to complete**
### 1. Implmement a dynamic set using bit vectors.

* You should implement the three primary set methods:
    * insert(x),
    * search(x),
    * delete(x).
* Your methods should run in O(1).
* This question is based on exercise 11.1-2 of chapter 11 in the CLRS textbook.

### 2. Implement a method to uniformly select a key at random from a hash map which uses chaining.

* A hint here is to use the maximum length of a chain to consider the chains as rows (or columns) of a 2D-array.
    * you can then randomly select a row and column and walk the list to the correct spot (if it exists)
    * if the position is empty, you can re-select a random row and column until it is not empty.
* This question is based on exercise 11.2-6 of chapter 11 in the CLRS textbook

### 3. Modify the chain hash map implementation to use a multiplicative hash function.

* This is based on the content on page 284-286 in section 11.3 of the CLRS textbook.
* Additionally, there is a discussion of this in Donald Knuth's *Art of Computer Programming Volume 3* in the chapter on searching.
* You should probably work the by-hand problem related from the paper homework to understand the hash function better.

### 4. Modify the probe hash map implementation to use Double Hashing.

* This is based on the discussion on page 295-296 in section 11.4 of the CLRS textbook.
* Page 295 gives the probe sequence to use based on the secondary hash function.
* Page 296 gives an example of the secondary hash function to use with our hash function.

### 5. Modify the Probe Hash Map implementation to not use the ```"__DELETED__"``` flag.

* This is based on the discussion in section 11.5 and pseudocode on page 303 of chapter 11 of the CLRS textbook.
* You will need to modify the insert function as well to accommodate this change.
* See page 302 for a discussion of the inverse probe function ```g(k,q)```.

<mark>___________________________________________________________________________________________________________</mark>

## Load the required notebooks

In [14]:
%%capture
%run Ch11_ChainHashMap.ipynb

In [15]:
%%capture
%run Ch11_ProbeHashMap.ipynb

## Exercises

<mark>_____________________________________________________________________________________</mark>

<mark>_____________________________________________________________________________________</mark>

### 1. Implmement a dynamic set using bit vectors

One way to store a bit vector is to use an integer variable for the "set" or "bit vector". Internally, this value is stored in binary and does not take additional space.

*example*

```self.set = 8``` would mean that set is stored in binary as $1000_2$ which we can interpret as the set $S=\{3\}$.
We can then work backwards to see how to add to such a set when stored as a (binary) number.

*example*
Supposed we want the set $S=\{1,3\}$, then we would have a bit-vector $1010_2$ which can be stored as the number ```self.set = 12```. The required operation to add element $1$ to the set $S$ was to add the value $2^1$ to the integer which is the set.

In this way, a $1$ in the binary representation of the set number tells us that the corresponding element is in the set.

This can used to implement in O(1) the required operations.

**HINT**
I left some code in the ```__str__``` print code to give an example of how I wrote my class.

In [73]:
class dynamicset:
    """a dynamic set class which uses a bit vector to keep the elements"""
    
    def __init__(self):
        self.set = 0            # use an integer for storage (bit vector)

    def insert(self,x):
        """insert element x into the set"""
        ###############################################################
        # insert your code here
        temp = self.set | 1 << x
        self.set = temp
        return

    def search(self,x):
        """search for element x in the set"""
        ###############################################################
        # insert your code here
        return (self.set & ( 1 << x)) != 0
    
    def delete(self,x):
        """deletes element x from the set"""
        ###############################################################
        # insert your code here
        temp = self.set & ~(1 << x)
        self.set = temp
        return

    def __str__(self):
        """printing utility"""
        out = "{"
        check = 1
        x = 0
        while check <= self.set:
            if self.set & check:
                out += str(x)+", "
            check = check<<1
            x += 1
        out += "}"
        return out

#### testing

In [74]:
S = dynamicset()
elements = []

N = 6
for i in range(N):
    e = random.randrange(32)
    elements.append(e)
    print("\ninserting element:",e)
    
    S.insert(e)
    print(S)

# shuffle element list and call delete
random.shuffle(elements)
for i in range(len(elements)//2):
    e = elements[i]
    print("\ndeleting element:",e)
    
    S.delete(e)
    print(S)


inserting element: 15
{15, }

inserting element: 2
{2, 15, }

inserting element: 23
{2, 15, 23, }

inserting element: 18
{2, 15, 18, 23, }

inserting element: 8
{2, 8, 15, 18, 23, }

inserting element: 15
{2, 8, 15, 18, 23, }

deleting element: 15
{2, 8, 18, 23, }

deleting element: 8
{2, 18, 23, }

deleting element: 18
{2, 23, }


In [75]:
S.search(16)

False

In [76]:
S.search(3)

False

In [12]:
S.search(33)

False

<mark>_____________________________________________________________________________________</mark>

### 2. Implement a method to uniformly select a key at random from a hash map which uses chaining.

The class below has been modified for your use in this problem. The textbook hints that we should use the max length of a chain and then consider the chain hash map as being in a 2D array were each list is a row or column of the array. Selection from this set of keys can be done by randomly selecting a row and column and walking to the appropriate linked-list value. There may or may not be enough nodes in the linked lists, so you will have to repeat this until you have success. This method should give roughly uniform sampling even though the separate chains will be different lengths.

Implement this method for the map below based on the ```self.max_length``` parameter which (hopefully) stores the max chain length for the table.

The test code at the bottom creates a map and then draws many random keys from the table. The dictionary counts the number of times each key is called. If the values of the dictionary are similar, then your algorithm is roughly uniform.

**Note**

Due to the law of large numbers, we cannot observe exactly even numbers on a random experiment. To observe this, we should look at larger and larger samples and observe the counts becoming more even. (i.e. count the number of heads flipped from a coin when taking n=10, n=100, n=1000, ...).

In [77]:
import random
class chainhashmapMAX(chainhashmap):
    """a hash map which implements separate chaining collision resolution

        modified to store the length of each chain in an array
        the max_length attribute saves the overall longest chain length
    """
        
    def __init__(self,capacity=11):
        """constructor for hash maps"""
        self.m = capacity # table size
        self.table = [linkedlist() for i in range(capacity)]
        self.lengths = [0 for i in range(capacity)]           # array of chain lengths
        self.n = 0        # number of elements
        self.max_length = max(self.lengths)
        
    def insert(self,x):
        """insert record x into the table"""
        hash_idx = self.h(x.key)
        L = self.table[hash_idx]          # extract appropriate linked-list   
        L.prepend(x)                      # insert new item
        self.lengths[hash_idx] += 1       # increase the length
        self.max_length = max(self.lengths)
    
    def delete(self,x):
        """delete item x from the table"""
        hash_idx = self.h(x.key)
        L = self.table[hash_idx]
        L.delete(x)
        self.lengths[hash_idx] -= 1       # increase the length
        self.max_length = max(self.lengths)

    def rand_key(self):
        """returns a random key from the map with ~uniform probability"""
        ###############################################################
        # insert your code here
        if self.max_length == 0:
            return None
        
        while True:
            row = random.randint(0,self.m -1)
            col = random.randint(0,self.max_length - 1)

            if col < self.lengths[row]:
                current = self.table[row].head

                for _ in range(col):
                    current = current.next
                return current.key.key



        
        
        
        
        
        
        return

#### testing

In [78]:
M = chainhashmapMAX(20)

KEYS = []
for i in range(50):
    x = chainhashmapMAX.item(rand_key(),random.randrange(99))
    KEYS.append(x.key)
    M.insert(x)
print("chain hash map:")
print(M)

print("\nlength of chains:")
print(M.lengths)

print("\nkeys stored:")
print(KEYS)

chain hash map:
(head)[(JRR,89), (SLY,21), (GXJ,0)](tail)
(head)[(VNH,36)](tail)
(head)[(CZZ,91), (TFI,13), (YTN,81), (XFF,83), (NMB,30), (VNQ,44)](tail)
(head)[(RMH,1), (EZT,35), (HJL,27), (PSR,71)](tail)
(head)[(MVW,31), (CVA,97), (PRG,19)](tail)
(head](tail)
(head)[(XSE,49), (BMH,21), (ZRU,76), (IBN,80), (EVL,35)](tail)
(head)[(XHW,83)](tail)
(head)[(ZQH,69), (FKW,96)](tail)
(head](tail)
(head)[(UWL,98), (VQU,30), (BYA,40), (KLO,97), (ALL,91), (BTK,21)](tail)
(head)[(WHD,36), (HJX,45)](tail)
(head)[(OLZ,56), (KKL,76)](tail)
(head)[(AJE,50), (LVD,88), (JKO,86), (GLO,10)](tail)
(head)[(WBR,4)](tail)
(head)[(DXA,37), (RHO,85), (HDP,11), (USP,45)](tail)
(head)[(MGI,67)](tail)
(head)[(EVV,67), (JIE,55)](tail)
(head](tail)
(head)[(QZU,80), (YQO,20), (DOY,35)](tail)


length of chains:
[3, 1, 6, 4, 3, 0, 5, 1, 2, 0, 6, 2, 2, 4, 1, 4, 1, 2, 0, 3]

keys stored:
['VNQ', 'BTK', 'HJX', 'GLO', 'USP', 'PRG', 'HDP', 'JIE', 'GXJ', 'PSR', 'NMB', 'HJL', 'JKO', 'XFF', 'LVD', 'EVL', 'ALL', 'KLO', 'BYA'

In [79]:
D = {}
for key in KEYS:
    D[key] = 0

for i in range(5000):
    random_key = M.rand_key()
    D[random_key] += 1
print(D)

{'VNQ': 103, 'BTK': 118, 'HJX': 102, 'GLO': 93, 'USP': 96, 'PRG': 76, 'HDP': 96, 'JIE': 117, 'GXJ': 113, 'PSR': 84, 'NMB': 110, 'HJL': 97, 'JKO': 93, 'XFF': 111, 'LVD': 93, 'EVL': 115, 'ALL': 80, 'KLO': 100, 'BYA': 105, 'DOY': 95, 'RHO': 101, 'IBN': 90, 'WBR': 95, 'YQO': 95, 'VQU': 100, 'DXA': 104, 'YTN': 122, 'AJE': 103, 'TFI': 94, 'CVA': 104, 'EZT': 89, 'SLY': 93, 'FKW': 100, 'VNH': 111, 'WHD': 95, 'UWL': 87, 'MGI': 94, 'ZQH': 109, 'ZRU': 111, 'CZZ': 113, 'QZU': 99, 'BMH': 109, 'XSE': 96, 'MVW': 86, 'KKL': 117, 'EVV': 110, 'JRR': 104, 'RMH': 82, 'XHW': 105, 'OLZ': 85}


In [80]:
max(D.values())

122

In [81]:
min(D.values())

76

<mark>_____________________________________________________________________________________</mark>

### 3. Modify the chain hash map implementation to use a multiplicative hash function according to section 11.3 in CLRS.

See page 284 of chapter 11. You should also do the corresponding paper homework problem on the multiplicative method. There is additionally an example at the bottom of page 285 to the top of page 286 that is informative.

The test code below creates the ```chainhashmap``` object with a set of random keys. The next test code creates the new ```chainmapbetterhash``` with the same set of keys so that you can compare how they look with the different hash functions.

In [91]:
class chainmapbetterhash(chainhashmap):
    """a sub-class of chainhashmap with a better hash function"""
    
    def __init__(self,capacity=16):
        """constructor for hash maps"""
        super().__init__(capacity=16)
    
    def h(self,key):
        """overload the hash function h from chainhashmap"""
        ###############################################################
        # insert your code here
        A = 0.6180339887
        temp = (self.m * ((hash(key) * A) % 1))
        temp = int(temp)
        
        return temp

#### testing

In [92]:
M = chainhashmap()

KEYS = []
for i in range(10):
    x = chainhashmap.item(rand_key(),random.randrange(99))
    KEYS.append(x.key)
    M.insert(x)
print("After Construction ...")
print(M)
print("\nTotal Keys:",KEYS)

for key in KEYS:
   x = M.search(key) # extract item
   M.delete(x)       # delete item
print("\n\nAfter deletion ...")
print(M)          # print modified table

After Construction ...
(head](tail)
(head](tail)
(head](tail)
(head)[(FUM,98)](tail)
(head)[(DDR,12), (CWB,61)](tail)
(head](tail)
(head)[(BGF,5), (UNL,92)](tail)
(head](tail)
(head)[(VCY,16)](tail)
(head)[(YPE,55), (USC,87), (TXA,83)](tail)
(head)[(XTB,72)](tail)


Total Keys: ['TXA', 'USC', 'CWB', 'FUM', 'YPE', 'VCY', 'UNL', 'BGF', 'DDR', 'XTB']


After deletion ...
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)



In [84]:
M = chainmapbetterhash()

for key in KEYS:
    x = chainmapbetterhash.item(key,random.randrange(99))
    M.insert(x)
print("After Construction ...")
print(M)
print("\nTotal Keys:",KEYS)

for key in KEYS:
   x = M.search(key) # extract item
   M.delete(x)       # delete item
print("\n\nAfter deletion ...")
print(M)          # print modified table

After Construction ...
(head)[(XOX,41), (KZU,78), (BTY,0), (PQC,82), (OQO,56), (VDO,17), (RPI,37), (IPL,31), (SJG,73), (LKF,83)](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)


Total Keys: ['LKF', 'SJG', 'IPL', 'RPI', 'VDO', 'OQO', 'PQC', 'BTY', 'KZU', 'XOX']


After deletion ...
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)
(head](tail)



<mark>_____________________________________________________________________________________</mark>

### 4. Modify the probe hash map implementation to use Double Hashing.

See the discussion on page 295 of the textbook. The main hash probe sequence is given here and can be used to modify the ```probehashmap``` code from the example (Feel free to copy-past this code and try to modify it to use double-hashing). I added the h2 method for the secondary hash function that I was referring to which is given on page 296 (or at least my interpretation of this text here).

I did not include the ```delete(self,q)``` method here since I left my version to use the delete that was from the ```probehashmap```.

In [ ]:
class doublehashmap(probehashmap):
    """a hash map which implements linear probing for collision resolution"""
    
    def h2(self,key):
        """suggested secondary hash function from page 296 in CLRS textbook"""
        return 1 + ( hash(key) % (self.m-1) )
    
    def insert(self,k):
        """insert key k into the table"""
        ###############################################################
        # insert your code here
        hash_idx_h1 = self.h(k)
        hash_idx_h2 = self.h2(k)

        i = 0
        while i < self.m:
            q = (hash_idx_h1 + (i) * hash_idx_h2) % self.m
            if self.table[q] in [None,"__DELETED__"]:
                self.table[q] = k        
                return q                 
            else:
                i = i + 1  

        if i==self.m:
            raise Exception("hash table overflow")
        
        return
    
    def search(self,k):
        """search the table for key"""
        ###############################################################
        # insert your code here
        hash_idx_h1 = self.h(k)
        hash_idx_h2 = self.h2(k)

        i = 0 
        while i < self.m:
            q = (hash_idx_h1 + (i) * hash_idx_h2) % self.m
            if self.table[q] == k:
                return q             
            elif self.table[q] == None:
                return None           
            else:
                i = i + 1  

        if i==self.m:
            return None
       
        return
        
        
        
        
        

#### testing

In [86]:
M = doublehashmap(101)

KEYS = []
for i in range(100):
    k = rand_key()
    KEYS.append(k)
    M.insert(k)
print(M)
#print("\nkeys = ",KEYS)

['IMJ', 'OLS', 'SSK', 'VUM', 'OCK', 'MEX', 'WTE', 'CDC', 'YMT', 'SSH', 'AGI', 'XOD', 'HHV', 'SRG', 'QLL', 'EEU', 'FQF', 'EWT', 'KCU', 'HSG', 'DPR', 'QHP', 'QEN', 'PIJ', 'DLI', 'TBR', 'KCS', 'OTT', 'FZG', 'UUE', 'ZUA', 'KGF', 'ERL', 'PCA', 'AXE', 'ESR', 'ZXF', 'JJD', 'YEX', None, 'VNB', 'ZLW', 'KJK', 'PUK', 'LYH', 'FFU', 'WSA', 'ISQ', 'KIZ', 'IFD', 'ISR', 'PDR', 'WOR', 'VYU', 'UPC', 'WGB', 'HES', 'RQB', 'WIL', 'OHF', 'TDQ', 'NSH', 'GQS', 'KRA', 'GCZ', 'UWK', 'NLD', 'LGI', 'WOP', 'XKK', 'GIC', 'VBT', 'LGG', 'GUW', 'XQY', 'AVX', 'EAE', 'MYC', 'IMA', 'FPL', 'FSD', 'CEQ', 'CPV', 'VYE', 'NVC', 'OFA', 'JZW', 'YFD', 'VNC', 'EZE', 'VSJ', 'VZD', 'XPI', 'RUJ', 'WDS', 'CUB', 'MLH', 'XCX', 'LSB', 'XQO', 'TCB']


In [87]:
for key in KEYS:
    x = M.search(key) # extract item
    M.delete(x)       # delete item
print(M)          # print modified table

['__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', None, '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__DELETED__', '__

<mark>_____________________________________________________________________________________</mark>

### 5. Modify the Probe Hash Map implementation to not use the ```"__DELETED__"``` flag.

See page 302 and 303 of section 11.5 in the textbook. The purpose of the section is for implementation notes on open-address hash maps and this is one of the notes. The ```"__DELETED__"``` flag is sort of a cheat and does not lead to the best performance in the long run of the hash table. To accommodate for this, you can move the keys in the table to "collapse" the probe sequences so that they do not get to long. This is my understanding of the pseudocode given on page 303.

You should implement this modified method in the framework of my ```probehashmap``` class.

In [ ]:
class probehashmapDELETE(probehashmap):
    """a hash map which implements linear probing for collision resolution"""
    
    def g(self,k,q):
        """deletion helper function from CLRS page 302"""
        return (q - self.h(k)) % self.m
    
    def insert(self,k):
        """insert key k into the table"""
        ###############################################################
        # insert your code here
        hash_idx = self.h(k)     #
        
        i = 0
        while i < self.m:
            q = (hash_idx + i) % self.m  
            if self.table[q] in [None,"__DELETED__"]:
                self.table[q] = k        
                return q                 
            else:
                i = i + 1            
        if i==self.m:
            raise Exception("hash table overflow")
        return
    
    def delete(self,q):
        """delete index q from the table"""
        ###############################################################
        # insert your code here
        while True:
            self.table[q] = None
            q_prime = q

            while True:
                q_prime = (q_prime + 1) % self.m
                k_prime = self.table[q_prime]
                if k_prime is None:
                    return
                
                if self.g(k_prime,q) < self.g(k_prime,q_prime):
                    break 
            
            self.table[q] = k_prime
            q = q_prime
                
        
        return

In [89]:
M = probehashmapDELETE(101)

KEYS = []
for i in range(100):
    k = rand_key()
    KEYS.append(k)
    M.insert(k)
print(M)
#print("\nkeys = ",KEYS)

['ZTW', 'ABB', 'KVM', 'WHZ', 'QWD', 'TEZ', 'PLE', 'YRN', 'IBT', 'EHM', 'PRL', 'TPC', 'MFO', 'CCC', 'JUT', 'ZWQ', 'RAF', 'HTE', 'VEI', 'NUE', 'SVW', 'RBZ', 'PTW', 'SLR', 'FYD', 'YAE', 'QDQ', 'OAB', 'OPL', 'UWP', 'LVH', 'LNH', 'VEY', 'ZVL', 'ZUG', 'NZG', 'RKD', 'KXK', 'ACS', 'WLL', 'AVY', 'BCA', 'BFG', 'BFD', 'MFX', 'STM', 'FVF', 'URR', 'OHK', 'LTR', 'VHY', 'CFT', 'UJD', 'XWI', 'FSL', 'GCN', 'YCC', 'DFO', 'BMY', 'YLC', 'MHJ', 'UAA', 'XDL', 'YYR', 'SOP', 'LTA', 'GAZ', 'BOI', 'LTS', 'VJS', 'UJT', 'OFY', 'XUZ', 'WKY', None, 'GBI', 'FOH', 'WHX', 'LGJ', 'RRL', 'SXN', 'NRQ', 'GQH', 'CVI', 'LHS', 'QVA', 'VYF', 'TTA', 'DFS', 'CAC', 'NYA', 'ZHD', 'UBT', 'DSW', 'FGL', 'MZX', 'WJO', 'YPK', 'QXD', 'GBU', 'XQO']


In [90]:
for key in KEYS:
    x = M.search(key) # extract item
    M.delete(x)       # delete item
print(M)          # print modified table

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
